In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
#1

def get_json(BASE_URL: str) -> dict:
    response = requests.get(BASE_URL)
    if response.status_code == 200: 
        return response.json()
    else: 
        raise Exception(f'Error, status_code {response.status_code}')

get_json(BASE_URL + 'status.json')['service']
#get_json(BASE_URL + 'orders/page-99.json')


'Northern Star Shop API'

In [0]:
#2
os.path.join(RAW_DIR,'orders/page-1.json')
def fetch_all(first_url: str) -> list:
    response = get_json(first_url)
    data = response['data']
    next_page = response['next']
    while next_page:
        response = get_json(urljoin(first_url, next_page))
        data += response['data']
        next_page = response['next']
    return data

fetch_all(BASE_URL + 'orders/page-1.json')


[{'order_id': 'NSS-00001',
  'order_date': '2026-08-15',
  'customer': {'name': 'NIKOLA STOJANOVIĆ', 'city': 'Zrenjanin'},
  'product_id': 1002,
  'quantity': 1,
  'unit_price': 6990.0,
  'payment': 'cash'},
 {'order_id': ' nss-00002 ',
  'order_date': '2026-08-15',
  'customer': {'name': ' Jovana Pavlović ', 'city': 'Čačak'},
  'product_id': 1094,
  'quantity': 3,
  'unit_price': 1290.0,
  'payment': 'card'},
 {'order_id': 'NSS-00003',
  'order_date': '2026-08-15',
  'customer': {'name': 'Sara Živković', 'city': 'Beograd'},
  'product_id': 1041,
  'quantity': 1,
  'unit_price': 6990.0,
  'payment': 'card'},
 {'order_id': 'NSS-00004',
  'order_date': '2026-08-15',
  'customer': {'name': 'Sara Đorđević', 'city': ' Subotica '},
  'product_id': 1075,
  'quantity': 1,
  'unit_price': 990.0,
  'payment': 'cash'},
 {'order_id': 'NSS-00005',
  'order_date': '2026-08-15',
  'customer': {'name': 'Stefan Popović', 'city': 'Niš'},
  'product_id': 1044,
  'quantity': 1,
  'unit_price': 1290.0,
  '

In [0]:
#3

def save_json(records: list, path: str) ->None:
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    return None

save_json([{'id': 1, 'name': 'Jovan'}], os.path.join(RAW_DIR, 'orders.json'))

In [0]:
#4

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status() 
    with open(path,'wb') as f:
        f.write(response.content)
    return len(response.content)

#download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
#download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))

In [0]:
#5
def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json'))
    teams = download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    products = download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    matches = download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))
    orders = pd.json_normalize(orders, sep='_')
    orders = orders.rename(columns= {'customer_name': 'customer', 'customer_city': 'city'})
    teams = pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    products = pd.read_csv(os.path.join(raw_dir, 'products.csv'))
    matches = pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))
    return {
        'orders': orders,
        'teams': teams,
        'products': products,
        'matches': matches
    }
extract(RAW_DIR)

{'orders':          order_id  order_date  product_id quantity unit_price payment  \
 0       NSS-00001  2026-08-15        1002        1     6990.0    cash   
 1      nss-00002   2026-08-15        1094        3     1290.0    card   
 2       NSS-00003  2026-08-15        1041        1     6990.0    card   
 3       NSS-00004  2026-08-15        1075        1      990.0    cash   
 4       NSS-00005  2026-08-15        1044        1     1290.0    card   
 ...           ...         ...         ...      ...        ...     ...   
 1847    NSS-01836  2026-11-15        1046        3     7490.0    card   
 1848    NSS-01837  2026-11-15        1024        1     1290.0    card   
 1849    NSS-01838  2026-11-15        1042        2     6490.0    cash   
 1850    NSS-01839  2026-11-15        1004        2     2290.0    cash   
 1851    NSS-01840  2026-11-15        1045        2     1490.0  paypal   
 
                customer        city  
 0     NIKOLA STOJANOVIĆ   Zrenjanin  
 1      Jovana Pavlovi

In [0]:
#6
def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset = 'product_id')
    df['team_code'] = df['team_code'].str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()
    df['price'] =  pd.to_numeric(df['price'].str.replace(' RSD', '', regex=False).str.replace(',', '.'))
    df['active'] = df['active'].str.upper().isin(['TRUE','YES'])
    return df

clean_products(pd.read_csv(os.path.join(RAW_DIR, 'products.csv')))


,product_id,sku,product_name,team_code,category,price,active
0,1001,NPM-JER-1001,NPM Home Jersey,NPM,jerseys,7490.0,True
1,1002,NPM-JER-1002,NPM Away Jersey,NPM,jerseys,6990.0,True
2,1003,NPM-SCA-1003,NPM Scarf,NPM,scarves,1490.0,True
3,1004,NPM-CAP-1004,NPM Cap,NPM,caps,2290.0,True
4,1005,NPM-ACC-1005,NPM Mug,NPM,accessories,990.0,True
...,...,...,...,...,...,...,...
100,1101,NSL-BAL-1101,NSL Match Ball,NaN,balls,7490.0,True
101,1102,NSL-BAL-1102,NSL Training Ball,NaN,balls,2990.0,True
102,1103,NSL-ACC-1103,NSL Water Bottle,NaN,accessories,890.0,True
103,1104,NSL-ACC-1104,NSL Backpack,NaN,accessories,3990.0,True


In [0]:
#7
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    if 'customer.city' in df.columns:
        df['city'] = df['customer.city']
    elif 'city' not in df.columns:
        df['city'] = 'Unknown'

    df['city'] = df['city'].fillna('Unknown').str.title()
    df['payment'] = df['payment'].str.upper()
    return df

clean_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')))

,order_id,order_date,customer,product_id,quantity,unit_price,payment,city
0,NSS-00001,2026-08-15,NaN,1002,1.0,6990.0,CASH,Unknown
1,NSS-00002,2026-08-15,NaN,1094,3.0,1290.0,CARD,Unknown
2,NSS-00003,2026-08-15,NaN,1041,1.0,6990.0,CARD,Unknown
3,NSS-00004,2026-08-15,NaN,1075,1.0,990.0,CASH,Unknown
4,NSS-00005,2026-08-15,NaN,1044,1.0,1290.0,CARD,Unknown
...,...,...,...,...,...,...,...,...
1847,NSS-01836,2026-11-15,NaN,1046,3.0,7490.0,CARD,Unknown
1848,NSS-01837,2026-11-15,NaN,1024,1.0,1290.0,CARD,Unknown
1849,NSS-01838,2026-11-15,NaN,1042,2.0,6490.0,CASH,Unknown
1850,NSS-01839,2026-11-15,NaN,1004,2.0,2290.0,CASH,Unknown


In [0]:
#8
def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
  unique = df.drop_duplicates(subset=['order_id'], keep='first')
  duplicates = df[df.duplicated(subset=['order_id'], keep=False)]
  return (unique, duplicates)
remove_duplicate_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')))



(         order_id  order_date  \
 0       NSS-00001  2026-08-15   
 1      nss-00002   2026-08-15   
 2       NSS-00003  2026-08-15   
 3       NSS-00004  2026-08-15   
 4       NSS-00005  2026-08-15   
 ...           ...         ...   
 1847    NSS-01836  2026-11-15   
 1848    NSS-01837  2026-11-15   
 1849    NSS-01838  2026-11-15   
 1850    NSS-01839  2026-11-15   
 1851    NSS-01840  2026-11-15   
 
                                                customer  product_id  quantity  \
 0     {'name': 'NIKOLA STOJANOVIĆ', 'city': 'Zrenjan...        1002       1.0   
 1        {'name': ' Jovana Pavlović ', 'city': 'Čačak'}        1094       3.0   
 2          {'name': 'Sara Živković', 'city': 'Beograd'}        1041       1.0   
 3       {'name': 'Sara Đorđević', 'city': ' Subotica '}        1075       1.0   
 4             {'name': 'Stefan Popović', 'city': 'Niš'}        1044       1.0   
 ...                                                 ...         ...       ...   
 1847   {'name':

In [0]:
#9
def validate_order(row: dict, product_ids: set) -> str:
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if not pd.to_numeric(row['quantity'], errors='coerce').is_integer() or row['quantity'] <= 0:
        return 'invalid_quantity'
    if not pd.to_numeric(row['unit_price'], errors='coerce').is_integer() or row['unit_price'] <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    return 'OK'

validate_order({'order_id': 'NSS-123', 'quantity': 1, 'unit_price': 100, 'product_id': '123'}, {'123', '456'})

'OK'

In [0]:
# 10
def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = clean_orders(df)
    df = df.merge(
        products[['product_id', 'product_name', 'team_code', 'category']],
        on='product_id',
        how='left'
    )
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce').fillna(0.0)
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    df['total'] = df['quantity'] * df['unit_price']
    return df
transform_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')), pd.read_csv(os.path.join(RAW_DIR, 'products.csv')))

,order_id,order_date,customer,product_id,quantity,unit_price,payment,city,product_name,team_code,category,total
0,NSS-00001,2026-08-15,NaN,1002,1,6990.0,CASH,Unknown,NPM Away Jersey,NPM,jerseys,6990.0
1,NSS-00002,2026-08-15,NaN,1094,3,1290.0,CARD,Unknown,BRU Cap,BRU,caps,3870.0
2,NSS-00003,2026-08-15,NaN,1041,1,6990.0,CARD,Unknown,RVA Home Jersey,RVA,jerseys,6990.0
3,NSS-00004,2026-08-15,NaN,1075,1,990.0,CASH,Unknown,LSB Mug,LSB,accessories,990.0
4,NSS-00005,2026-08-15,NaN,1044,1,1290.0,CARD,Unknown,RVA Cap,RVA,caps,1290.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1900,NSS-01836,2026-11-15,NaN,1046,3,7490.0,CARD,Unknown,GMW Home Jersey,gmw,jerseys,22470.0
1901,NSS-01837,2026-11-15,NaN,1024,1,1290.0,CARD,Unknown,KLT Cap,KLT,caps,1290.0
1902,NSS-01838,2026-11-15,NaN,1042,2,6490.0,CASH,Unknown,RVA Away Jersey,RVA,jerseys,12980.0
1903,NSS-01839,2026-11-15,NaN,1004,2,2290.0,CASH,Unknown,NPM Cap,NPM,caps,4580.0


In [0]:
#11
def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    valid = transform_orders(df, products)
    valid, duplicates = remove_duplicate_orders(valid)
    rejected = pd.concat([valid[valid.apply(lambda row: validate_order(row, set(products['product_id'])), axis=1) != 'OK'], duplicates])
    rejected['reason'] = rejected.apply(lambda row: validate_order(row, set(products['product_id'])) if row['order_id'] not in duplicates['order_id'].values else 'duplicate', axis=1)
    return (valid, rejected)

split_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')), pd.read_csv(os.path.join(RAW_DIR, 'products.csv')))





(         order_id order_date  customer  product_id  quantity  unit_price  \
 0       NSS-00001 2026-08-15       NaN        1002         1      6990.0   
 1      NSS-00002  2026-08-15       NaN        1094         3      1290.0   
 2       NSS-00003 2026-08-15       NaN        1041         1      6990.0   
 3       NSS-00004 2026-08-15       NaN        1075         1       990.0   
 4       NSS-00005 2026-08-15       NaN        1044         1      1290.0   
 ...           ...        ...       ...         ...       ...         ...   
 1900    NSS-01836 2026-11-15       NaN        1046         3      7490.0   
 1901    NSS-01837 2026-11-15       NaN        1024         1      1290.0   
 1902    NSS-01838 2026-11-15       NaN        1042         2      6490.0   
 1903    NSS-01839 2026-11-15       NaN        1004         2      2290.0   
 1904    NSS-01840 2026-11-15       NaN        1045         2      1490.0   
 
      payment     city     product_name team_code     category    total  


In [0]:
#12

def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    codes = dict(zip(teams['team_id'], teams['code']))
    played = matches[matches['status'] == 'finished']
    home = pd.DataFrame(
    {
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].dt.normalize(),
        'team_code': played['home_team_id'].map(codes),
        'opponent_code': played['away_team_id'].map(codes),
        'goals_for': played['home_goals'],
        'goals_against': played['away_goals'],
        'goal_diff': played['home_goals'] - played['away_goals']
    })
    away = pd.DataFrame(
    {
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].dt.normalize(),
        'team_code': played['away_team_id'].map(codes),
        'opponent_code': played['home_team_id'].map(codes),
        'goals_for': played['away_goals'],
        'goals_against': played['home_goals'],
        'goal_diff': played['away_goals'] - played['home_goals']
    })
    return pd.concat([home, away])
team_results(pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet')), pd.read_csv(os.path.join(RAW_DIR, 'teams.csv')))

,match_id,match_date,team_code,opponent_code,goals_for,goals_against,goal_diff
0,2627001,2026-08-15,BRU,FWR,2,1,1
1,2627002,2026-08-15,CHS,VDC,0,1,-1
2,2627003,2026-08-15,MBA,WFT,3,0,3
3,2627004,2026-08-15,NPM,IVU,1,0,1
4,2627005,2026-08-15,PCU,HGA,1,1,0
...,...,...,...,...,...,...,...
115,2627086,2026-10-24,NPM,KLT,3,1,2
116,2627087,2026-10-24,RVA,WFT,2,0,2
117,2627088,2026-10-25,CHS,PCU,1,3,-2
118,2627089,2026-10-25,IVU,FWR,0,2,-2


In [0]:
#13
def league_table(results: pd.DataFrame) -> pd.DataFrame:
    points = results.groupby('team_code').agg(
        played = ('match_id', 'count'),
        won = ('goals_for', lambda x: (x > x.shift(1)).sum()),
        drawn = ('goals_for', lambda x: (x == x.shift(1)).sum()),
        lost = ('goals_for', lambda x: (x < x.shift(1)).sum()),
        goals_for = ('goals_for', 'sum'),
        goals_against = ('goals_against', 'sum'),
        goal_diff = ('goals_for', lambda x: x.sum() - x.shift(1).sum())
    )
    points['points'] = 3 * points['won'] + points['drawn']
    points = points.sort_values(['points', 'goal_diff', 'goals_for'], ascending=False)
    points['position'] = range(1, len(points) + 1)
    return points.reset_index()[['position', 'team_code', 'played', 'won', 'drawn', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points']]
league_table(team_results(pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet')), pd.read_csv(os.path.join(RAW_DIR, 'teams.csv'))))


,position,team_code,played,won,drawn,lost,goals_for,goals_against,goal_diff,points
0,1,CHS,12,6,1,4,15,20,1,19
1,2,MBA,12,5,2,4,15,15,2,17
2,3,STH,12,4,5,2,17,15,1,17
3,4,VDC,12,4,4,3,18,13,2,16
4,5,HGA,11,4,4,2,13,6,2,16
5,6,IVU,12,5,1,5,17,13,0,16
6,7,OKT,11,4,4,2,6,10,0,16
7,8,NPM,12,4,3,4,15,15,3,15
8,9,GMW,12,4,3,4,14,16,2,15
9,10,WFT,12,4,3,4,16,19,1,15


In [0]:
#14
def match_windows(results: pd.DataFrame) -> pd.DataFrame:
    results['date'] = results['match_date']
    results['window'] = 'W'
    windows = []
    for d in range(4):
        window = results.copy()
        window['date'] = window['date'] + pd.DateOffset(days=d)
        window['window'] = 'W' + str(d)
        windows.append(window)
    return pd.concat(windows, ignore_index=True)[['team_code', 'date', 'window']]
match_windows(team_results(pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet')), pd.read_csv(os.path.join(RAW_DIR, 'teams.csv'))))


,team_code,date,window
0,BRU,2026-08-15,W0
1,CHS,2026-08-15,W0
2,MBA,2026-08-15,W0
3,NPM,2026-08-15,W0
4,PCU,2026-08-15,W0
...,...,...,...
947,NPM,2026-10-27,W3
948,RVA,2026-10-27,W3
949,CHS,2026-10-28,W3
950,IVU,2026-10-28,W3


In [0]:
#15
def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df['revenue'] = df['quantity'] * df['unit_price']
    return df.groupby(column).agg({'revenue': 'sum'}).sort_values('revenue', ascending=False)
revenue_by(transform_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')), pd.read_csv(os.path.join(RAW_DIR, 'products.csv'))), 'team_code')
revenue_by(transform_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')), pd.read_csv(os.path.join(RAW_DIR, 'products.csv'))), 'category')

,revenue
category,
jerseys,4864800.0
jerseys,1237110.0
scarves,975090.0
caps,929460.0
JERSEYS,766880.0
accessories,445230.0
balls,354870.0
JERSEYS,263560.0
accessories,247810.0


,team_id,code,team_name,city,founded_year
0,1,NPM,Northport Mariners,Northport,1894
1,2,IVU,Ironvale United,Ironvale,1902
2,3,FWR,Frostwick Rovers,Frostwick,1889
3,4,HGA,Harrowgate Athletic,Harrowgate,1911
4,5,KLT,Kelda Town,Kelda,1897
5,6,STH,Stormhaven FC,Stormhaven,1920
6,7,BMC,Brightmoor City,Brightmoor,1905
7,8,AFV,Ashford Vale,Ashford,1899
8,9,RVA,Ravenscar Albion,Ravenscar,1908
9,10,GMW,Glenmouth Wanderers,Glenmouth,1884


In [0]:
#17
def sales_after_results(valid: pd.DataFrame,windows: pd.DataFrame) -> pd.DataFrame:
    valid['revenue'] = valid['quantity'] * valid['unit_price']
    valid = valid.groupby(['team_code', 'order_date'])['revenue'].sum().reset_index(name='revenue')
    calendar = pd.merge(pd.DataFrame({'team_code': windows['team_code'].unique()}), pd.DataFrame({'order_date': valid['order_date'].unique()}), how='cross')
    calendar = pd.merge(calendar, valid, left_on=['team_code', 'order_date'], right_on=['team_code', 'order_date'], how='left')
    calendar['revenue'] = calendar['revenue'].fillna(0)
    calendar = pd.merge(calendar, windows, left_on=['team_code', 'order_date'], right_on=['team_code', 'date'], how='left')
    calendar['window'] = calendar['window'].fillna('none')
    return calendar.groupby('window').agg({'order_date': 'count', 'revenue': 'sum'}).rename(columns={'order_date': 'team_days', 'revenue': 'revenue'}).sort_values('revenue', ascending=False).reset_index().assign(avg_daily_revenue=lambda x: x['revenue'] / x['team_days']).round(2)
sales_after_results(transform_orders(pd.read_json(os.path.join(RAW_DIR, 'orders.json')), pd.read_csv(os.path.join(RAW_DIR, 'products.csv'))), match_windows(team_results(pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet')), pd.read_csv(os.path.join(RAW_DIR, 'teams.csv')))))

,window,team_days,revenue,avg_daily_revenue
0,none,954,3569520.0,3741.64
1,W0,238,1366910.0,5743.32
2,W1,232,1098520.0,4735.00
3,W3,218,1010590.0,4635.73
4,W2,218,918940.0,4215.32


In [0]:
#17
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
        written[name] = len(df)
    return written


In [0]:
#18
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric', 'value']).to_csv(os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary
run_pipeline(RAW_DIR, PROCESSED_DIR)

/home/spark-a80465ea-6910-4f1d-912a-6d/.ipykernel/75/command-5834401452668973-3432975337:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['team_code'] = df['team_code'].str.strip().str.upper()
/home/spark-a80465ea-6910-4f1d-912a-6d/.ipykernel/75/command-5834401452668973-3432975337:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['category'] = df['category'].str.strip().str.lower()
/home/spark-a80465ea-6910-4f1d-912a-6d/.ipykernel/75/command-5834401452668973-3432975337:6: SettingWithCopyWarning: 
A val

{'total_rows': 1852,
 'duplicates': 24,
 'valid_rows': 1840,
 'rejected_rows': 141,
 'total_revenue': 10112310.0,
 'files_written': 8}